<p align="left"><img src="https://symbolica.io/icon.svg" width="150px"></p>

## Setup

In [ ]:
!pip install --upgrade symbolica
from IPython.display import Markdown as md
from symbolica import *

# get a license easily at https://symbolica.io/docs/get_started.html#license
set_license_key('gcolab-demo-key-do-not-use-elsewhere')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 12.3 MB/s eta 0:00:00


## Introduction


Symbolica is a new computer algebra system that aims to handle expressions with billions of terms, while at the same time being easy to use. It has APIs for Python, Rust and C++.

In this Jupyter notebook we will explore some features of Symbolica.

--------------

Declare a variable and a function

In [ ]:
x = Expression.var('x')
f = Expression.fun('f')

Create a new expression by composing variables, functions, and numbers

In [ ]:
e = f(x) + 4
print(e)

f(x)+4


Expressions can also be parsed, with liberal parsing rules

In [ ]:
e = Expression.parse('f(x,y)+ 5 (x + y)^2 + 1_123_456')
print(e)

5*(x+y)^2+f(x,y)+1123456


The output format can be easily changed

In [ ]:
md(e.to_latex())

$$5 \left(x+y\right)^{2}+f\!\left(x,y\right)+1123456$$

## Pattern matching

Replace all occurrences of $x$ in $x f(x, 1, y^x)$ by 5

In [ ]:
e = Expression.parse('x*f(x,1,y^x)')
e = e.replace_all(x, 5)
md(e.to_latex())

$$5 f\!\left(5,1,y^{5}\right)$$

Use a wildcard, a variable ending in an underscore `_` to match to any subexpression

In [ ]:
x_ = Expression.var('x_')

e = f(5).replace_all(f(x_), f(x_ + 1))
md(e.to_latex())

$$f\!\left(6\right)$$

Repeated wildcards must match to the same subexpressions

In [ ]:
y_, z_, w_ = Expression.vars('y_', 'z_', 'w_')

e = f(1,2,3,2,4).replace_all(f(x_,y_,z_,y_,w_), f(x_,z_,w_))
md(e.to_latex())

$$f\!\left(1,3,4\right)$$

Wildcards can match to multiple ranges / subexpressions. Use double underscores for a wildcard that has an unbounded maximum length.

In [ ]:
x__ = Expression.var('x__')
e = f(1,2,3,4).replace_all(f(x__,4), f(x__))
md(e.to_latex())

$$f\!\left(1,2,3\right)$$

Conditions can be set on wildcards

In [ ]:
e = (f(1,4) + f(1,2,3,4)).replace_all(f(x__,4), f(x__), x__.req_len(2, None)) # minimum length is 2
md(e.to_latex())

$$f\!\left(1,4\right)+f\!\left(1,2,3\right)$$

Conditions can contain logical operations and user-defined functions

In [ ]:
e = (f(1)*f(2)*f(3)).replace_all(f(x_), 1, x_.req_type(AtomType.Num) & x_.req(lambda m: m == 2 or m == 3))
md(e.to_latex())

$$f\!\left(1\right)$$

Matches can be iterated over, like the `regex` module in Python

In [ ]:
from symbolica import Expression

x, y, z, n_, x_ = Expression.vars('x', 'y', 'z', 'n_', 'x_')
e = x**2*(y+z) + x**3*(y+z**2)

# match returns a dictionary that maps every wildcard to its value
coeff_list = [(m[n_], m[x_]) for m in e.match(x**n_*x_)]
for (pow, content) in coeff_list:
    print(x**pow, content)

x^2 y+z
x^3 y+z^2


### Transformers

The right-hand side can be used to apply arbitrarily complex operations after the wildcard substitution. For example:

In [ ]:
e = f((x+1)**2).replace_all(f(x_), f(x_.transform().expand()))
md(e.to_latex())


$$f\!\left(2 x+x^{2}+1\right)$$

Custom functions can be used as well

In [ ]:
e = f(2).replace_all(f(x_), x_.transform().map(lambda r: r**2))
md(e.to_latex())

$$4$$

Below is a more complicated example from particle physics, that computes the color of the triangle-box-triangle in QCD

In [ ]:
nc, i, x___, y___, z___, w___, a_ = Expression.vars(
    'n_c', 'i', 'x___', 'y___', 'z___', 'w___', 'a_')
T, f, trace = Expression.funs('T', 'f', 'trace')

# tri-box-tri
e = T(1, 12)*T(12, 11)*T(11, 13, 10)*T(10, 9)*T(9, 15, 8)*T(8, 7) * \
    T(7, 6)*T(6, 5)*T(5, 15, 4)*T(4, 3)*T(3, 13, 2)*T(2, 1)

# collect the color string into a trace
e = e.replace_all(T(x___, a_)*T(a_, w___), T(x___, w___), repeat=True)
e = e.replace_all(T(a_, x___, a_), trace(x___))

last_e = T()
while last_e != e:
    last_e = e
    e = e.replace_all(trace(), nc)
    e = e.replace_all(trace(a_), 0)
    e = e.replace_all(trace(x___, a_, z___, a_, w___), trace(
        x___, w___)*trace(z___) / 2 - trace(x___, z___, w___) / 2 / nc)
    e = e.replace_all(f(x___, a_, z___), -2 * i *
                      trace(x___, a_, z___) + 2 * i * trace(z___, a_, x___))
    e = e.replace_all(trace(x___, a_, z___) * trace(w___, y___, a_), trace(x___,
                      a_, w___, z___) / 2 - trace(x___, z___) * trace(w___, a_) / 2 / nc)

md(e.to_rational_polynomial().to_latex())

$$\frac{1-2n_c^{2}+n_c^{4}}{4n_c}$$

Instead of calling several Python instructions in a row, it is faster to use *transformers* that execute commands in Rust.

In [ ]:
# tri-box-tri
e = T(1, 12)*T(12, 11)*T(11, 13, 10)*T(10, 9)*T(9, 15, 8)*T(8, 7) * \
    T(7, 6)*T(6, 5)*T(5, 15, 4)*T(4, 3)*T(3, 13, 2)*T(2, 1)

# collect the color string into a trace
e = e.replace_all(T(x___, a_)*T(a_, w___), T(x___, w___), repeat=True)
e = e.replace_all(T(a_, x___, a_), trace(x___))

e = e.transform().repeat(
    Transformer()
    .replace_all(trace(), nc)
    .replace_all(trace(a_), 0)
    .replace_all(trace(x___, a_, z___, a_, w___),
                 trace(x___, w___) * trace(z___) / 2 - trace(x___, z___, w___) / 2 / nc)
    .replace_all(f(x___, a_, z___), -2 * i *
                 trace(x___, a_, z___) + 2 * i * trace(z___, a_, x___))
    .replace_all(trace(x___, a_, z___) * trace(w___, y___, a_),
                 trace(x___, a_, w___, z___) / 2 - trace(x___, z___) * trace(w___, a_) / 2 / nc)
).execute()
md(e.to_rational_polynomial().to_latex())

$$\frac{1-2n_c^{2}+n_c^{4}}{4n_c}$$

### Programmable patterns

Python can be used to make programmable patterns. In the following example we will apply it to compute a trace using the following simplification algoritm:


$$Tr(\mu_1,\ldots,\mu_n) = \sum_{k=2}^n (-1)^k g(\mu_1,\mu_k) Tr(\mu_2,\ldots,\mu_{k-1},\mu_{k+1},\ldots \mu_n)$$

In [ ]:
from symbolica import Expression

x__ = Expression.var('x__')
p = Expression.vars(*['p' + str(i + 1) for i in range(10)])
p_ = Expression.vars(*['p' + str(i + 1) + '_' for i in range(10)])
f = Expression.fun('f')
d = Expression.fun('d', is_symmetric=True)

e = f(*p[:6])

# find the maximum chain length
max_length = 0
for m in e.match(f(x__)):
    max_length = max(max_length, len(m[x__]))

for l in range(max_length,0,-1):
    if l % 2 == 1:
        e = e.replace_all(f(*p_[:l]), 0) # odd trace is 0
    else:
        e = e.replace_all(f(*p_[:l]), sum((-1)**(k+1) * d(p_[0], p_[k]) * f(*p_[1:k], *p_[k+1:l]) for k in range(1,l))).expand()
e = e.replace_all(f(), 4)
print('\t ' + e.pretty_str(terms_on_new_line=True))

	 4*d(p1,p2)*d(p3,p4)*d(p5,p6)
	-4*d(p1,p2)*d(p3,p5)*d(p4,p6)
	+4*d(p1,p2)*d(p3,p6)*d(p4,p5)
	-4*d(p1,p3)*d(p2,p4)*d(p5,p6)
	+4*d(p1,p3)*d(p2,p5)*d(p4,p6)
	-4*d(p1,p3)*d(p2,p6)*d(p4,p5)
	+4*d(p1,p4)*d(p2,p3)*d(p5,p6)
	-4*d(p1,p4)*d(p2,p5)*d(p3,p6)
	+4*d(p1,p4)*d(p2,p6)*d(p3,p5)
	-4*d(p1,p5)*d(p2,p3)*d(p4,p6)
	+4*d(p1,p5)*d(p2,p4)*d(p3,p6)
	-4*d(p1,p5)*d(p2,p6)*d(p3,p4)
	+4*d(p1,p6)*d(p2,p3)*d(p4,p5)
	-4*d(p1,p6)*d(p2,p4)*d(p3,p5)
	+4*d(p1,p6)*d(p2,p5)*d(p3,p4)


## Features

### Derivatives

In [ ]:
x, y = Expression.vars('x', 'y')
e = 1/(x*y+1)
e = e.derivative(x)
md(e.to_latex())

$$-y \left(x y+1\right)^{-2}$$

### Series expansion

Symbolica can also produce the Taylor series:

In [ ]:
x, y = Expression.vars('x', 'y')
f = Expression.fun('f')

e = 2* x**2 * y + f(x)
e = e.taylor_series(x, 0, 2)

print(e)

f(0)+x*der(1,f(0))+1/2*x^2*(4*y+der(2,f(0)))


### Collecting

You can iterate over all coefficients of an expression viewed as a polynomial in a variable:

In [ ]:
x, y = Expression.vars('x', 'y')
e = 5*x + x * y + x**2 + 5

l = e.coefficient_list(x)
for key, value in l:
    print(key, value)

x y+5
x^2 1
1 5




You can also collect:
```
print(e.collect(x))
```
yielding
```
x^2+x*(y+5)+5
```

You can also provide a user-defined function to map the key and one to map the coefficient. For example:

In [ ]:
from symbolica import Expression

x, y = Expression.vars('x', 'y')
var, coeff = Expression.funs('var', 'coeff')

e = 5*x + x * y + x**2 + 5

e = e.collect(x, key_map=lambda x: var(x), coeff_map=lambda x: coeff(x))
print(e)

5*var(1)+[5+y]*var(x)+var(x^2)


### Linear system solving

In [ ]:
from symbolica import Expression

x, y, c = Expression.vars('x', 'y', 'c')
f = Expression.fun('f')

x_r, y_r = Expression.solve_linear_system(
    [f(c)*x + y + c, y + c**2], [x, y])
print('x =', x_r, ', y =', y_r)

x = (-c+c^2)*f(c)^-1 , y = -c^2


### Numerical evaluation

Expression can be numerically evaluated by specifying the values for all user-defined variables and function:

In [ ]:
from symbolica import Expression

x, f = Expression.vars('x', 'f')
e = Expression.parse('cos(x)*3 + f(x,2)')

a = e.evaluate({x: 1}, {f: lambda args: args[0]+args[1]})
print(a)

4.620906917604419


Evaluation with complex numbers is also supported:

In [ ]:
from symbolica import Expression

x, y, f = Expression.vars('x', 'y', 'f')
e = Expression.parse('sqrt(x)*cos(x)*sin(x)*exp(x)*x^2 / y + f(2)')
print(e.evaluate_complex({x: 1 + 2j, y: 4 + 3j}, {f: lambda x: x[0].real}))

(-45.19905714494966-29.187180407418648j)


### Parallel term processing

Symbolica can process a huge expression one term at a time, in parallel. This means that the expression does not have to be in memory (similar to FORM).

In [ ]:
from symbolica import Transformer

e = ((1+x)**10).expand()
r = e.map(Transformer().replace_all(x, 6))
print(r)

282475249


## Polynomials

Symbolic has world-class rational polynomial arithmetic. Polynomials can easily be constructed:

In [ ]:
from symbolica import Expression, Polynomial
x, y = Expression.vars('x', 'y')
e = ((x + y + x**2)**3).expand()
p = e.to_polynomial() + Polynomial.parse('x^2*y + 1 + 2*x + y^4', ['x', 'y'])
md(p.to_latex())

$$1+y^{3}+y^{4}+2x+3xy^{2}+4x^{2}y+3x^{2}y^{2}+x^{3}+6x^{3}y+3x^{4}+3x^{4}y+3x^{5}+x^{6}$$

Rational polynomial arithmetic is supported too:

In [ ]:
from symbolica import RationalPolynomial
p = RationalPolynomial.parse('1/x+(1+x)^2/(1+x+y)', ['x', 'y'])
md(p.to_latex())

$$\frac{1+y+2x+2x^{2}+x^{3}}{x+xy+x^{2}}$$

Computing the greatest common divisor of polynomials is an expensive operation. Below Symbolica does it in a few seconds, whereas other computer algebra packages take more than a minute.

In [ ]:
a = Expression.parse('(1 + 3*x1 + 5*x2 + 7*x3 + 9*x4 + 11*x5 + 13*x6 + 15*x7)^7 - 1').to_rational_polynomial()
b = Expression.parse('(1 - 3*x1 - 5*x2 - 7*x3 + 9*x4 - 11*x5 - 13*x6 + 15*x7)^7 + 1').to_rational_polynomial()
g = Expression.parse('(1 + 3*x1 + 5*x2 + 7*x3 + 9*x4 + 11*x5 + 13*x6 - 15*x7)^7 + 3').to_rational_polynomial()
ag = a * g
bg = b * g
gcd = ag.gcd(bg) - g
print(gcd)

0


### Factorization

You can compute the square-free factorization of a multivariate polynomial:

In [ ]:
from symbolica import Expression

p = Expression.parse('3*(2*x^2+y)(x^3+y)^2(1+4*y)^2(1+x)').expand().to_polynomial()

print('Square-free factorization of {}:'.format(p))
for f, exp in p.factor_square_free():
    print('\t({})^{}'.format(f, exp))

Square-free factorization of 3*y^3+24*y^4+48*y^5+3*x*y^3+24*x*y^4+48*x*y^5+6*x^2*y^2+48*x^2*y^3+96*x^2*y^4+12*x^3*y^2+96*x^3*y^3+192*x^3*y^4+6*x^4*y^2+48*x^4*y^3+96*x^4*y^4+12*x^5*y+96*x^5*y^2+192*x^5*y^3+15*x^6*y+120*x^6*y^2+240*x^6*y^3+3*x^7*y+24*x^7*y^2+48*x^7*y^3+6*x^8+48*x^8*y+96*x^8*y^2+6*x^9+48*x^9*y+96*x^9*y^2:
	(1+4*y)^2
	(1+x)^1
	(y+2*x^2)^1
	(y+x^3)^2
	(3)^1


Or compute the complete factorization over the rationals:

In [ ]:
p = Expression.parse(
    '(x*(2+2*y+2*z)+1)*(x*(4+z^2)+y+3)*(x*(w+w^2+4+y)+w+5)').expand().to_polynomial()

print('Factorization of {}:'.format(p))
for f, exp in p.factor():
    print('\t({})^{}'.format(f, exp))

Factorization of 15+3*w+5*y+y*w+62*x+30*x*z+5*x*z^2+13*x*w+6*x*w*z+x*w*z^2+3*x*w^2+47*x*y+10*x*y*z+9*x*y*w+2*x*y*w*z+x*y*w^2+11*x*y^2+2*x*y^2*w+80*x^2+64*x^2*z+14*x^2*z^2+10*x^2*z^3+18*x^2*w+14*x^2*w*z+3*x^2*w*z^2+2*x^2*w*z^3+10*x^2*w^2+6*x^2*w^2*z+x^2*w^2*z^2+82*x^2*y+14*x^2*y*z+11*x^2*y*z^2+16*x^2*y*w+2*x^2*y*w*z+2*x^2*y*w*z^2+8*x^2*y*w^2+2*x^2*y*w^2*z+16*x^2*y^2+2*x^2*y^2*z+2*x^2*y^2*w+2*x^2*y^2*w^2+2*x^2*y^3+32*x^3+32*x^3*z+8*x^3*z^2+8*x^3*z^3+8*x^3*w+8*x^3*w*z+2*x^3*w*z^2+2*x^3*w*z^3+8*x^3*w^2+8*x^3*w^2*z+2*x^3*w^2*z^2+2*x^3*w^2*z^3+40*x^3*y+8*x^3*y*z+10*x^3*y*z^2+2*x^3*y*z^3+8*x^3*y*w+2*x^3*y*w*z^2+8*x^3*y*w^2+2*x^3*y*w^2*z^2+8*x^3*y^2+2*x^3*y^2*z^2:
	(1+2*x+2*x*z+2*x*y)^1
	(3+y+4*x+x*z^2)^1
	(5+w+4*x+x*w+x*w^2+x*y)^1


### Groebner basis
A Groebner basis of a polynomial system can be computed using lexicographical ordering or reverse graded lexicographical order (grevlex). The latter is often much faster.

In [ ]:
from symbolica import Polynomial
basis = Polynomial.groebner_basis(
    [Expression.parse("a b c d - 1").to_polynomial(),
     Expression.parse("a b c + a b d + a c d + b c d").to_polynomial(),
     Expression.parse("a b + b c + a d + c d").to_polynomial(),
     Expression.parse("a + b + c + d").to_polynomial()],
    grevlex=False,
    print_stats=True
)
for p in basis:
    print(p)

b+a+c+d
-2*b^2+2*a*b-a^2+2*a^2*b^4+2*c*b+c^2
b^2-a*b+a^2-a^2*b^4-c*b+c*a
-a+a*b^4-c+c*b^4
-b-a+a^2*b^3+a^3*b^2
1-b^4-a^2*b^2+a^2*b^6


### Fast evaluation

Symbolica can write a polynomial in an optimized form that allows for fast evaluation. It can even produce C++ output that evaluates the polynomial.

In [ ]:
from symbolica import Expression
x, y = Expression.vars('x', 'y')
e = ((x + y + x**2)**3).expand().to_polynomial()
eval = e.optimize() # to_file='test2.cpp'

print(eval.evaluate([[1., 2.], [3. ,4.]]))

[64.0, 4096.0]


If chosen to generate a file, it produces output such as:

```cpp
#include <iostream>

template<typename T>
T evaluate(T x,T y) {
	T Z0,Z1,Z2,Z3,Z4,Z5,Z6,Z7;
	Z0 = x;
	Z1 = y;
	Z2 = T(3);
	Z3 = T(1);
	Z4 = T(6);
	Z5 = Z0+Z2;
	Z5 = Z0*Z5;
	Z6 = Z1+Z3;
	Z6 = Z2*Z6;
	Z5 = Z5+Z6;
	Z5 = Z0*Z5;
	Z7 = Z1*Z4;
	Z5 = Z3+Z5+Z7;
	Z5 = Z0*Z5;
	Z6 = Z1*Z6;
	Z5 = Z5+Z6;
	Z5 = Z0*Z5;
	Z6 = Z1*Z1;
	Z7 = Z2*Z6;
	Z5 = Z5+Z7;
	Z5 = Z0*Z5;
	Z6 = Z1*Z6;
	Z5 = Z5+Z6;
	return Z5;
}

int main() {
	std::cout << evaluate<double>(0.25,0.5) << std::endl;
}
```

## Numerical integration

Symbolica uses the Vegas algorithm to numerically integrate in a continuous grid

In [ ]:
from symbolica import NumericalIntegrator

def integrand(samples):
    res = []
    for sample in samples:
        res.append(sample.c[0]**2+sample.c[1]**2)
    return res

avg, err, chi_sq = NumericalIntegrator.continuous(2).integrate(integrand, min_error = 0.0007)

### Multi-channeling

Symbolica supports multi-channeling, by allowing to nest discrete layers and continuous grids.

In [ ]:
from symbolica import NumericalIntegrator, Sample
from math import sin, cos
def integrand(samples):
    res = []
    for sample in samples:
        if sample.d[0] == 0:
            res.append(sin(sample.c[0]))
        else:
            res.append(cos(sample.c[0]))
    return res

integrator = NumericalIntegrator.discrete(
    [NumericalIntegrator.continuous(1), NumericalIntegrator.continuous(1)])
avg, err, chi_sq = integrator.integrate(integrand, min_error = 0.003)

### More control

You can also do the sample-update loop yourself to have full control over the integration (and its parallelization)

In [ ]:
def integrand(samples):
    res = []
    for sample in samples:
        res.append(sample.c[0]**2+sample.c[1]**2)
    return res

integrator = NumericalIntegrator.continuous(2)
rng = RandomNumberGenerator(5, 0)

for i in range(10):
    samples = integrator.sample(10000 + i * 1000, rng)
    res = integrand(samples)
    integrator.add_training_samples(samples, res)
    avg, err, chi_sq = integrator.update(1.5)
    print('Iteration {}: {:.6} +- {:.6}, chi={:.3}'.format(i + 1, avg, err, chi_sq))

Iteration 1: 0.66517 +- 0.00419967, chi=0.0
Iteration 2: 0.668933 +- 0.00194499, chi=0.511
Iteration 3: 0.665846 +- 0.00119609, chi=1.69
Iteration 4: 0.666665 +- 0.000866449, chi=1.51
Iteration 5: 0.667088 +- 0.000695759, chi=1.35
Iteration 6: 0.667071 +- 0.000590961, chi=1.12
Iteration 7: 0.667017 +- 0.000518758, chi=0.967
Iteration 8: 0.667025 +- 0.000464474, chi=0.846
Iteration 9: 0.667049 +- 0.000422298, chi=0.754
Iteration 10: 0.667047 +- 0.000388372, chi=0.678
